**Import pandas library**

In [1]:
# Import the pandas library and use the conventional alias 'pd'.
import pandas as pd

**Read the dataset**

In [2]:
# Read the final analysis-ready CSV file into a DataFrame named df.
df = pd.read_csv('final_dataset.csv')

In [3]:
df

,CustomerID,Age,Gender,Region,Income,Months_Subscribed,Total_Spend,Customer_Service_Calls,Rating_Score,Payment_Method,...,Income_Invalid_Flag,Rating_Was_Missing,High_Spend_Outlier_Flag,Avg_Monthly_Spend,Tenure_Group,Service_Intensity,Low_Rating_Flag,Churn_Label,Retention_Offer_Sent,Offer_Accepted
0,CUST-00001,45,Female,North,64406,42,2407.72,4,2.5,PayPal,...,0,0,0,57.33,37+ months,High (3+),1,Active,1,1
1,CUST-00002,38,Male,East,63664,56,2420.02,3,3.9,Credit Card,...,0,0,0,43.21,37+ months,High (3+),0,Active,0,0
2,CUST-00003,47,Male,West,30108,17,1545.50,0,5.0,PayPal,...,0,0,0,90.91,13-24 months,No calls,0,Active,1,0
3,CUST-00004,58,Male,North,66747,31,2055.80,3,5.0,Credit Card,...,0,0,0,66.32,25-36 months,High (3+),0,Active,0,0
4,CUST-00005,37,Female,North,93320,53,2369.31,0,3.8,PayPal,...,0,0,0,44.70,37+ months,No calls,0,Active,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,CUST-04996,39,Female,North,107527,30,1713.08,1,3.0,Credit Card,...,0,0,0,57.10,25-36 months,Low (1-2),0,Active,0,0
4996,CUST-04997,48,Male,North,25041,15,1369.50,1,4.9,Credit Card,...,0,0,0,91.30,13-24 months,Low (1-2),0,Active,1,1
4997,CUST-04998,77,Female,West,57367,53,1290.02,0,4.3,Credit Card,...,0,0,0,24.34,37+ months,No calls,0,Active,0,0
4998,CUST-04999,49,Male,North,87394,15,1970.70,3,3.5,Credit Card,...,0,0,0,131.38,13-24 months,High (3+),0,Churned,1,0


**Basic structural checks**

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 21 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   CustomerID               5000 non-null   object 
 1   Age                      5000 non-null   int64  
 2   Gender                   5000 non-null   object 
 3   Region                   5000 non-null   object 
 4   Income                   5000 non-null   int64  
 5   Months_Subscribed        5000 non-null   int64  
 6   Total_Spend              5000 non-null   float64
 7   Customer_Service_Calls   5000 non-null   int64  
 8   Rating_Score             5000 non-null   float64
 9   Payment_Method           5000 non-null   object 
 10  Churn                    5000 non-null   int64  
 11  Income_Invalid_Flag      5000 non-null   int64  
 12  Rating_Was_Missing       5000 non-null   int64  
 13  High_Spend_Outlier_Flag  5000 non-null   int64  
 14  Avg_Monthly_Spend       

In [5]:
# Display basic structural checks before statistical analysis begins.
print('Shape:', df.shape)                                             # Print the number of rows and columns in the DataFrame.
print('Total missing values:', df.isna().sum().sum())                 # Calculate and print the total count of all missing values in the DataFrame.
print('Duplicate CustomerIDs:', df['CustomerID'].duplicated().sum())  # Count and print the number of duplicate CustomerID entries.

Shape: (5000, 21)
Total missing values: 0
Duplicate CustomerIDs: 0


**Active vs churn customers**

In [6]:
# Display the number of active and churned customers.
print('Churn status counts:') # Print a descriptive header for the churn status counts.
print(df['Churn_Label'].value_counts()) # Count and display the occurrences of each unique value in the 'Churn_Label' column.

Churn status counts:
Active     4782
Churned     218
Name: Churn_Label, dtype: int64


In [7]:
import matplotlib.pyplot as plt
import seaborn as sns

**Mean, standard deviation, and standard error**

In [13]:
# Select the Age column and store it in a shorter variable named age.
age = df['Age'] # Extract the 'Age' column from the DataFrame 'df' and assign it to the variable 'age'.

# Count how many customer ages are available in the sample.
n = len(age) # Calculate the total number of entries (sample size) in the 'age' Series.

# Calculate the sample mean: the average age of the 5,000 observed customers.
mean_age = age.mean() # Compute the arithmetic mean of the 'age' Series.

# Calculate the sample standard deviation: variation among individual ages.
sd_age = age.std() # Compute the standard deviation of the 'age' Series, indicating data dispersion.

# Import SciPy's statistics functions.
from scipy import stats # Import the 'stats' module from the SciPy library for statistical functions.

# Calculate the standard error of the sample mean.
# This estimates how much the mean would vary across repeated samples.
se_age = stats.sem(age) # Calculate the standard error of the mean for the 'age' Series.

# Print the four quantities in a readable form.
print('Sample size:', n) # Display the calculated sample size.
print(f'Sample mean Age: {mean_age:.2f} years') # Display the sample mean of age, formatted to two decimal places.
print(f'Sample standard deviation: {sd_age:.2f} years') # Display the sample standard deviation of age, formatted to two decimal places.
print(f'Standard error of mean Age: {se_age:.3f} years') # Display the standard error of the mean for age, formatted to three decimal places.

Sample size: 5000
Sample mean Age: 39.75 years
Sample standard deviation: 11.56 years
Standard error of mean Age: 0.163 years


Large SD means customer values are widely spread around the mean. Small SE means the estimated mean is relatively precise.

**Group comparison: Two groups - Independent samples t-test**

In [14]:
# Select Rating_Score for active customers (Churn = 0).
active = df.loc[df['Churn'] == 0, 'Rating_Score'] # Filters the DataFrame to select 'Rating_Score' for customers where 'Churn' is 0 (active).

# Select Rating_Score for churned customers (Churn = 1).
churned = df.loc[df['Churn'] == 1, 'Rating_Score'] # Filters the DataFrame to select 'Rating_Score' for customers where 'Churn' is 1 (churned).

# Run Welch's independent-samples t-test.
# equal_var=False tells SciPy not to assume equal population variances.
t_stat, p_value = stats.ttest_ind(active, churned, equal_var=False) # Performs Welch's t-test comparing 'Rating_Score' between active and churned customers, not assuming equal variances.

# Display the main inferential results.
print(f'Mean difference (Active - Churned): {active.mean() - churned.mean():.3f}') # Prints the difference between the mean 'Rating_Score' of active and churned customers, formatted to three decimal places.
print(f'Welch t statistic: {t_stat:.3f}') # Prints the calculated t-statistic from the Welch's t-test, formatted to three decimal places.
print(f'p-value: {p_value:.3e}') # Prints the p-value from the Welch's t-test, formatted in scientific notation with three decimal places.

Mean difference (Active - Churned): 0.469
Welch t statistic: 7.311
p-value: 4.203e-12


The p-value is far below 0.05. There is strong statistical evidence that average satisfaction ratings differ between active and churned customers. Active customers rate Zephyr about 0.47 points higher on average.

**Group comparison: Three or more groups - One-way ANOVA**

The choice of the type of one-way ANOVA will depend on the homogeneity of variance accross the groups. If the equal-variance condition is reasonably supported: use classical one-way ANOVA, otherwise use Welch's one-way ANOVA.

In [15]:
# Summarize Total_Spend separately for each Region before testing.
region_summary = df.groupby('Region')['Total_Spend'].agg(['count','mean','std','median']) # Group the DataFrame by 'Region' and calculate the count, mean, standard deviation, and median of 'Total_Spend' for each region.

# Round the output to two decimal places for readability.
print(region_summary.round(2)) # Print the 'region_summary' DataFrame, with all numerical values rounded to two decimal places.

        count     mean     std   median
Region                                 
East     1206  1627.30  673.24  1619.12
North    1269  1603.74  683.98  1607.25
South    1319  1619.32  690.33  1629.60
West     1206  1656.20  690.41  1639.06


Levene's test examines the equality of variance in such case.

In [16]:
# Create one Total_Spend array for each Region.
regional_groups = [group['Total_Spend'].values # Extracts 'Total_Spend' values for each regional group.
                   for _, group in df.groupby('Region')] # Iterates through groups created by grouping the DataFrame 'df' by 'Region'.

# Run Levene's test.
# center='median' makes the test less sensitive to non-normal data and outliers.
levene_stat, levene_p = stats.levene(*regional_groups, center='median') # Performs Levene's test for equality of variances across the regional groups, using the median for robustness.

# Display the result.
print(f"Levene's statistic: {levene_stat:.3f}") # Prints the calculated Levene's test statistic, formatted to three decimal places.
print(f'p-value: {levene_p:.3f}') # Prints the p-value from Levene's test, formatted to three decimal places.

Levene's statistic: 0.213
p-value: 0.887


For Total_Spend across Region, Levene's p-value was 0.887. Because p >= 0.05, the equal-variance condition is reasonably supported for this exercise. We can therefore use the classical one-way ANOVA.

In [17]:
# Import the one-way ANOVA function from statsmodels.
from statsmodels.stats.oneway import anova_oneway # Imports the anova_oneway function from the statsmodels library.

# Run the classical one-way ANOVA.
# use_var='equal' means the procedure assumes equal population variances.
anova_equal = anova_oneway(
    data=df['Total_Spend'], # Specifies the data column to be analyzed.
    groups=df['Region'], # Specifies the grouping variable.
    use_var='equal' # Assumes equal variances across groups for the classical ANOVA.
)

# Display the main results.
print(f'F statistic: {anova_equal.statistic:.3f}') # Prints the calculated F-statistic, formatted to three decimal places.
print(f'Numerator df: {anova_equal.df_num:.0f}') # Prints the numerator degrees of freedom, formatted to zero decimal places.
print(f'Denominator df: {anova_equal.df_denom:.0f}') # Prints the denominator degrees of freedom, formatted to zero decimal places.
print(f'p-value: {anova_equal.pvalue:.3f}') # Prints the p-value, formatted to three decimal places.

F statistic: 1.272
Numerator df: 3
Denominator df: 4996
p-value: 0.282


The ANOVA p-value is 0.282, which is greater than 0.05. At the 5% significance level, we do not reject the null hypothesis of equal regional population mean spending.

consider a second business question: does average Rating_Score differ across customer income levels? For this demonstration, we divide Income into four approximately equal-sized groups called quartiles.

In [20]:
# Create four income groups of approximately equal size.
# Q1 contains the lowest-income quarter of customers and Q4 the highest-income quarter.
df['Income_Band'] = pd.qcut( # Creates a new column 'Income_Band' by discretizing 'Income' into quartiles.
    df['Income'], # Specifies the 'Income' column as the data to be binned.
    q=4, # Divides the data into 4 quantiles (quartiles).
    labels=['Q1', 'Q2', 'Q3', 'Q4'] # Assigns descriptive labels to the four income bands.
)

# Summarize Rating_Score within each income band.
income_rating_summary = df.groupby('Income_Band', observed=False)['Rating_Score'].agg( # Groups the DataFrame by 'Income_Band' and calculates aggregate statistics for 'Rating_Score'.
    ['count', 'mean', 'std', 'median'] # Specifies the aggregation functions: count, mean, standard deviation, and median.
)
print(income_rating_summary.round(3)) # Prints the summary DataFrame, with numerical values rounded to three decimal places for readability.

             count   mean    std  median
Income_Band                             
Q1            1250  3.925  0.822     4.0
Q2            1250  3.917  0.876     4.0
Q3            1250  3.879  0.863     4.0
Q4            1250  3.910  0.833     4.0


We now use Levene's test to decide which ANOVA is more appropriate.

In [19]:
# Create one Rating_Score array for each Income_Band.
income_rating_groups = [group['Rating_Score'].values # Extracts 'Rating_Score' values for each income group.
                        for _, group in df.groupby('Income_Band', observed=False)] # Iterates through groups created by grouping the DataFrame 'df' by 'Income_Band'.

# Test the equal-variance condition.
levene2_stat, levene2_p = stats.levene(*income_rating_groups, center='median') # Performs Levene's test for equality of variances across the income groups, using the median for robustness.

print(f"Levene's statistic: {levene2_stat:.3f}") # Prints the calculated Levene's test statistic, formatted to three decimal places.
print(f'p-value: {levene2_p:.3f}') # Prints the p-value from Levene's test, formatted to three decimal places.

Levene's statistic: 3.311
p-value: 0.019


The p-value is 0.019, which is below 0.05. Unlike the regional spending example, here we find evidence that the group variances are not equal. Therefore, we do not use the equal-variance ANOVA as our main test. We move to Welch's ANOVA.

In [22]:
# Run Welch's one-way ANOVA because Levene's test indicated unequal variances.
welch_anova = anova_oneway( # Calls the anova_oneway function to perform Welch's ANOVA.
    data=df['Rating_Score'], # Specifies the 'Rating_Score' column as the dependent variable.
    groups=df['Income_Band'], # Specifies the 'Income_Band' column as the grouping variable.
    use_var='unequal' # Indicates that unequal variances across groups should be assumed for the ANOVA calculation.
)

# Display Welch's F statistic, adjusted degrees of freedom, and p-value.
print(f"Welch F statistic: {welch_anova.statistic:.3f}") # Prints the calculated Welch's F statistic, formatted to three decimal places.
print(f'Numerator df: {welch_anova.df_num:.0f}') # Prints the numerator degrees of freedom for Welch's ANOVA, formatted to zero decimal places.
print(f'Denominator df: {welch_anova.df_denom:.3f}') # Prints the denominator degrees of freedom for Welch's ANOVA, formatted to three decimal places.
print(f'p-value: {welch_anova.pvalue:.3f}') # Prints the p-value from Welch's ANOVA, formatted to three decimal places.

Welch F statistic: 0.698
Numerator df: 3
Denominator df: 2774.761
p-value: 0.553


The p-value is 0.553, which is greater than 0.05. Welch's ANOVA does not find evidence that their population mean Rating_Scores differ.

**Fit the multiple linear regression model**

In [26]:
df

,CustomerID,Age,Gender,Region,Income,Months_Subscribed,Total_Spend,Customer_Service_Calls,Rating_Score,Payment_Method,...,Rating_Was_Missing,High_Spend_Outlier_Flag,Avg_Monthly_Spend,Tenure_Group,Service_Intensity,Low_Rating_Flag,Churn_Label,Retention_Offer_Sent,Offer_Accepted,Income_Band
0,CUST-00001,45,Female,North,64406,42,2407.72,4,2.5,PayPal,...,0,0,57.33,37+ months,High (3+),1,Active,1,1,Q2
1,CUST-00002,38,Male,East,63664,56,2420.02,3,3.9,Credit Card,...,0,0,43.21,37+ months,High (3+),0,Active,0,0,Q2
2,CUST-00003,47,Male,West,30108,17,1545.50,0,5.0,PayPal,...,0,0,90.91,13-24 months,No calls,0,Active,1,0,Q1
3,CUST-00004,58,Male,North,66747,31,2055.80,3,5.0,Credit Card,...,0,0,66.32,25-36 months,High (3+),0,Active,0,0,Q2
4,CUST-00005,37,Female,North,93320,53,2369.31,0,3.8,PayPal,...,0,0,44.70,37+ months,No calls,0,Active,1,1,Q4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,CUST-04996,39,Female,North,107527,30,1713.08,1,3.0,Credit Card,...,0,0,57.10,25-36 months,Low (1-2),0,Active,0,0,Q4
4996,CUST-04997,48,Male,North,25041,15,1369.50,1,4.9,Credit Card,...,0,0,91.30,13-24 months,Low (1-2),0,Active,1,1,Q1
4997,CUST-04998,77,Female,West,57367,53,1290.02,0,4.3,Credit Card,...,0,0,24.34,37+ months,No calls,0,Active,0,0,Q1
4998,CUST-04999,49,Male,North,87394,15,1970.70,3,3.5,Credit Card,...,0,0,131.38,13-24 months,High (3+),0,Churned,1,0,Q3


In [24]:
# Define a regression formula for Total_Spend.
# C(...) tells statsmodels to treat those variables as categorical.
linear_formula = ( # Defines the linear regression formula as a multi-line string.
    'Total_Spend ~ Age + Income + Months_Subscribed + ' # Specifies 'Total_Spend' as the dependent variable and lists continuous independent variables.
    'Customer_Service_Calls + Rating_Score + ' # Continues listing continuous independent variables.
    'C(Gender) + C(Region) + C(Payment_Method)' # Specifies categorical independent variables, wrapped in C() for proper handling by statsmodels.
)
linear_formula

'Total_Spend ~ Age + Income + Months_Subscribed + Customer_Service_Calls + Rating_Score + C(Gender) + C(Region) + C(Payment_Method)'

In [25]:
# Import statsmodels' formula interface.
import statsmodels.formula.api as smf # Imports the formula API from statsmodels for defining regression models using R-style formulas.

# Estimate the ordinary least squares (OLS) regression model.
linear_model = smf.ols(linear_formula, data=df).fit() # Creates and fits an Ordinary Least Squares (OLS) regression model using the defined formula and the DataFrame 'df'.

# Print the complete statistical summary.
print(linear_model.summary()) # Prints a comprehensive summary of the regression model's results, including coefficients, R-squared, p-values, etc.

                            OLS Regression Results                            
Dep. Variable:            Total_Spend   R-squared:                       0.471
Model:                            OLS   Adj. R-squared:                  0.470
Method:                 Least Squares   F-statistic:                     370.2
Date:                Fri, 18 Sep 2026   Prob (F-statistic):               0.00
Time:                        16:06:18   Log-Likelihood:                -38147.
No. Observations:                5000   AIC:                         7.632e+04
Df Residuals:                    4987   BIC:                         7.640e+04
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
                                       coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------------
Intercep

<h3>Reference Table: Interpreting the Multiple Linear Regression Results</h3>

<table border="1" cellpadding="7" cellspacing="0" width="100%" style="table-layout:fixed;">
  <tr>
    <th width="18%">Critical observation</th>
    <th width="10%">Result</th>
    <th width="8%">p-value</th>
    <th width="14%">Statistical decision at 5% level</th>
    <th width="50%">Interpretation</th>
  </tr>

  <tr>
    <td><b>R-squared</b></td>
    <td><b>0.471</b></td>
    <td>—</td>
    <td>—</td>
    <td>
      The model explains <b>47.1% of the variation in Total_Spend</b><br>
      among the 5,000 customers. The remaining variation is associated<br>
      with factors not included in the model and random variation.
    </td>
  </tr>

  <tr>
    <td><b>Adjusted R-squared</b></td>
    <td><b>0.470</b></td>
    <td>—</td>
    <td>—</td>
    <td>
      After adjusting for the number of predictors, the model still explains<br>
      about <b>47.0% of the variation</b> in Total_Spend. Its closeness to<br>
      R-squared suggests little inflation from adding predictors.
    </td>
  </tr>

  <tr>
    <td><b>Overall F-test</b></td>
    <td><b>F = 370.2</b></td>
    <td><b>&lt; .001</b></td>
    <td><b>Reject H₀</b></td>
    <td>
      The regression model is statistically significant overall.<br>
      Taken together, the predictors provide statistically significant<br>
      explanatory information about Total_Spend.
    </td>
  </tr>

  <tr>
    <td><b>Intercept</b></td>
    <td>71.5165</td>
    <td>.174</td>
    <td>Not significant</td>
    <td>
      Predicted Total_Spend when all numerical predictors equal zero<br>
      and categorical predictors are at their reference categories.<br>
      It has limited practical meaning here.
    </td>
  </tr>

  <tr>
    <td>Gender: Male</td>
    <td>+25.0240</td>
    <td>.084</td>
    <td>Not significant</td>
    <td>
      Males are estimated to spend about <b>25.02 more</b> than females,<br>
      holding other predictors constant, but the difference is not<br>
      statistically significant.
    </td>
  </tr>

  <tr>
    <td>Gender: Non-Binary</td>
    <td>+41.9500</td>
    <td>.205</td>
    <td>Not significant</td>
    <td>
      Non-Binary customers are estimated to spend about <b>41.95 more</b><br>
      than females, controlling for other predictors, but the difference<br>
      is not statistically significant.
    </td>
  </tr>

  <tr>
    <td>Region: North</td>
    <td>+15.4673</td>
    <td>.441</td>
    <td>Not significant</td>
    <td>
      North customers are estimated to spend about <b>15.47 more</b><br>
      than East customers, holding other predictors constant,<br>
      but the difference is not statistically significant.
    </td>
  </tr>

  <tr>
    <td>Region: South</td>
    <td>+23.0561</td>
    <td>.246</td>
    <td>Not significant</td>
    <td>
      South customers are estimated to spend about <b>23.06 more</b><br>
      than East customers, holding other predictors constant,<br>
      but the difference is not statistically significant.
    </td>
  </tr>

  <tr>
    <td>Region: West</td>
    <td>+23.9716</td>
    <td>.238</td>
    <td>Not significant</td>
    <td>
      West customers are estimated to spend about <b>23.97 more</b><br>
      than East customers, holding other predictors constant,<br>
      but the difference is not statistically significant.
    </td>
  </tr>

  <tr>
    <td>Payment Method: Credit Card</td>
    <td>−13.3297</td>
    <td>.478</td>
    <td>Not significant</td>
    <td>
      Credit Card users are estimated to spend about <b>13.33 less</b><br>
      than Bank Transfer users, holding other predictors constant,<br>
      but the difference is not statistically significant.
    </td>
  </tr>

  <tr>
    <td>Payment Method: PayPal</td>
    <td>−22.2679</td>
    <td>.276</td>
    <td>Not significant</td>
    <td>
      PayPal users are estimated to spend about <b>22.27 less</b><br>
      than Bank Transfer users, controlling for other predictors,<br>
      but the difference is not statistically significant.
    </td>
  </tr>

  <tr>
    <td><b>Age</b></td>
    <td><b>+10.1526</b></td>
    <td><b>&lt; .001</b></td>
    <td><b>Significant</b></td>
    <td>
      Holding other predictors constant, each additional year of age<br>
      is associated with approximately <b>10.15 higher Total_Spend</b>.
    </td>
  </tr>

  <tr>
    <td><b>Income</b></td>
    <td><b>+0.0053</b></td>
    <td><b>&lt; .001</b></td>
    <td><b>Significant</b></td>
    <td>
      Holding other predictors constant, each additional income unit<br>
      is associated with about <b>0.0053 higher Total_Spend</b>.<br>
      A 10,000-unit increase corresponds to roughly <b>53 additional units</b>.
    </td>
  </tr>

  <tr>
    <td><b>Months Subscribed</b></td>
    <td><b>+25.3644</b></td>
    <td><b>&lt; .001</b></td>
    <td><b>Significant</b></td>
    <td>
      Holding other predictors constant, each additional subscription month<br>
      is associated with approximately <b>25.36 higher Total_Spend</b>.
    </td>
  </tr>

  <tr>
    <td>Customer Service Calls</td>
    <td>+6.7219</td>
    <td>.249</td>
    <td>Not significant</td>
    <td>
      Each additional service call is associated with an estimated<br>
      <b>6.72 higher Total_Spend</b>, but the relationship is not<br>
      statistically significant.
    </td>
  </tr>

  <tr>
    <td>Rating Score</td>
    <td>−7.2816</td>
    <td>.381</td>
    <td>Not significant</td>
    <td>
      A one-point increase in Rating_Score is associated with an estimated<br>
      <b>7.28 lower Total_Spend</b>, but the relationship is not<br>
      statistically significant.
    </td>
  </tr>

</table>

<p><b>Reference categories:</b> Female for Gender, East for Region, and Bank Transfer for Payment Method.</p>